## Config stuff

In [4]:

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()


In [5]:
spark = cc.startLocalCluster("DIM_DATE",4)
spark.getActiveSession()

# Creating Date dimension from scratch

## Step 1: Generate rows for a sequence of dates


In [16]:
from pyspark.sql.functions import *

beginDate = '2009-01-01'
endDate = '2023-12-31'

df_SQL = spark.sql(f"select explode(sequence(to_date('{beginDate}'), to_date('{endDate}'), interval 1 day)) as calendarDate, monotonically_increasing_id() as dateSK ")


df_SQL.createOrReplaceTempView('neededDates' )

spark.sql("select * from neededDates").show()

+------------+------+
|calendarDate|dateSK|
+------------+------+
|  2009-01-01|     0|
|  2009-01-02|     1|
|  2009-01-03|     2|
|  2009-01-04|     3|
|  2009-01-05|     4|
|  2009-01-06|     5|
|  2009-01-07|     6|
|  2009-01-08|     7|
|  2009-01-09|     8|
|  2009-01-10|     9|
|  2009-01-11|    10|
|  2009-01-12|    11|
|  2009-01-13|    12|
|  2009-01-14|    13|
|  2009-01-15|    14|
|  2009-01-16|    15|
|  2009-01-17|    16|
|  2009-01-18|    17|
|  2009-01-19|    18|
|  2009-01-20|    19|
+------------+------+
only showing top 20 rows



## Step 2: Create all typical dimension fields

### Method a: Use spark.sql to perform all the transformations with the help of a sql-query.
For many, creating an SQL-select statement is the most easy way to perform the transformation.

In [15]:
dimDate = spark.sql("select dateSK, \
  CalendarDate, \
  year(calendarDate) AS CalendarYear, \
  date_format(calendarDate, 'MMMM') as CalendarMonth, \
  month(calendarDate) as MonthOfYear, \
  date_format(calendarDate, 'EEEE') as CalendarDay, \
  dayofweek(calendarDate) AS DayOfWeek, \
  weekday(calendarDate) + 1 as DayOfWeekStartMonday, \
  case \
    when weekday(calendarDate) < 5 then 'Y' \
    else 'N' \
  end as IsWeekDay, \
  dayofmonth(calendarDate) as DayOfMonth, \
  case \
    when calendarDate = last_day(calendarDate) then 'Y' \
    else 'N' \
  end as IsLastDayOfMonth, \
  dayofyear(calendarDate) as DayOfYear, \
  quarter(calendarDate) as QuarterOfYear \
from  \
  neededDates \
order by \
  calendarDate")

dimDate.show()

+------+------------+------------+-------------+-----------+-----------+---------+--------------------+---------+----------+----------------+---------+-------------+
|dateSK|CalendarDate|CalendarYear|CalendarMonth|MonthOfYear|CalendarDay|DayOfWeek|DayOfWeekStartMonday|IsWeekDay|DayOfMonth|IsLastDayOfMonth|DayOfYear|QuarterOfYear|
+------+------------+------------+-------------+-----------+-----------+---------+--------------------+---------+----------+----------------+---------+-------------+
|     0|  2009-01-01|        2009|      January|          1|   Thursday|        5|                   4|        Y|         1|               N|        1|            1|
|     1|  2009-01-02|        2009|      January|          1|     Friday|        6|                   5|        Y|         2|               N|        2|            1|
|     2|  2009-01-03|        2009|      January|          1|   Saturday|        7|                   6|        N|         3|               N|        3|            1|
|   

* ```spark.sql``` is used to create the select query and returns the desired DataFrame. Remember to look-up the possible functions [here](https://spark.apache.org/docs/latest/api/sql/).
* ```dimDate.show()``` s used to show the records in a DataFrame. Use it during development, but disable when not needed anymore


### Method b: Use the dataframe API

This method does not use the sql-like language. You can achieve the same with this method and you get better code completion. See [DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html)
As an example two columns where added with   ```withColumn``` .
```èxpr()``` is used to write a snippet of 'sql' code and parse it into a column.

In [5]:
#from pyspark.sql.functions import explode, expr, sequence,col, date_format
df_SparkSQL = df_SQL \
    .withColumn("year", date_format("calendarDate",'yyyy')) \
    .withColumn("month", date_format("calendarDate",'MMMM')) \
    .withColumn("lasyDayOfMonth" \
                ,expr("case when calendarDate = last_day(calendarDate) then 'Y' \
                else 'N' \
                end as IsLastDayOfMonth"))
df_SparkSQL.show()

+------------+------+----+-------+--------------+
|calendarDate|dateSK|year|  month|lasyDayOfMonth|
+------------+------+----+-------+--------------+
|  2009-01-01|     0|2009|January|             N|
|  2009-01-02|     1|2009|January|             N|
|  2009-01-03|     2|2009|January|             N|
|  2009-01-04|     3|2009|January|             N|
|  2009-01-05|     4|2009|January|             N|
|  2009-01-06|     5|2009|January|             N|
|  2009-01-07|     6|2009|January|             N|
|  2009-01-08|     7|2009|January|             N|
|  2009-01-09|     8|2009|January|             N|
|  2009-01-10|     9|2009|January|             N|
|  2009-01-11|    10|2009|January|             N|
|  2009-01-12|    11|2009|January|             N|
|  2009-01-13|    12|2009|January|             N|
|  2009-01-14|    13|2009|January|             N|
|  2009-01-15|    14|2009|January|             N|
|  2009-01-16|    15|2009|January|             N|
|  2009-01-17|    16|2009|January|             N|


> ## TASK:
> Complete the transformation in method b until the result matches the result of method a.

# Step 3: Writing the data to a delta-file

In [6]:
dimDate.write.format("delta").mode("overwrite").saveAsTable("dimDate")


In [20]:
spark.stop()

NameError: name 'spark' is not defined